In [1]:
from nn_framework import *
import numpy as np
import pandas as pd

In [2]:
def create_sample_csv(filename="train.csv", n_samples=300):
    # Генерируем случайные признаки
    X = np.random.randn(n_samples, 3)

    # Генерируем целевую переменную (3 класса)
    # Класс зависит от суммы признаков для наглядности
    y = np.zeros(n_samples, dtype=int)
    sums = np.sum(X, axis=1)
    y[sums < -1] = 0
    y[(sums >= -1) & (sums <= 1)] = 1
    y[sums > 1] = 2

    # Создаем DataFrame
    df = pd.DataFrame(X, columns=['f1', 'f2', 'f3'])

    # Делаем One-Hot Encoding для таргета (превращаем 0 в [1,0,0], 1 в [0,1,0] и т.д.)
    # Это нужно для нашей функции CrossEntropy
    targets = pd.get_dummies(y, prefix='target')
    df = pd.concat([df, targets], axis=1)

    df.to_csv(filename, index=False)
    print(f"Файл {filename} успешно создан!")
    print(df.head())

In [3]:
create_sample_csv()

Файл train.csv успешно создан!
         f1        f2        f3  target_0  target_1  target_2
0  1.587889  1.347454 -0.412648     False     False      True
1  1.457970 -0.228135 -0.452660     False      True     False
2  0.123704  0.629018 -0.142450     False      True     False
3  0.491577 -0.649466  0.835259     False      True     False
4 -0.474056  1.129923 -0.354194     False      True     False


In [4]:
target_columns = ['target_0', 'target_1', 'target_2']
dataset = Dataset.from_csv("train.csv", target_cols=target_columns)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

# 3. Создаем нейросеть
# Вход: 3 признака, Выход: 3 класса
net = Sequential(
    Linear(3, 16),
    ReLU(),
    Linear(16, 3),
    Softmax()
)

# 4. Настраиваем обучение
optimizer = Adam(net.parameters(), lr=0.01)
# Оборачиваем в клиппинг градиентов
optimizer = GradientClipping(optimizer, clip_value=1.0)

model = Model(
    network=net,
    loss_fn=CrossEntropy(),
    optimizer=optimizer,
    metrics=[Accuracy()]
)

# 5. Обучаем
print("Начинаем обучение...")
model.fit(loader, epochs=20)

# 6. Проверяем на одном примере
test_x = [[0.5, 0.8, 0.1]]
prediction = model.predict(test_x)
print(f"\nПредсказание для {test_x}:")
print(f"Вероятности классов: {prediction}")
print(f"Выбранный класс: {np.argmax(prediction)}")

Начинаем обучение...
Epoch [1/20] - loss: 1.0179 - ACCURACY: 0.7033 - 0.01s
Epoch [2/20] - loss: 0.6424 - ACCURACY: 0.8133 - 0.01s
Epoch [3/20] - loss: 0.4869 - ACCURACY: 0.8900 - 0.01s
Epoch [4/20] - loss: 0.3838 - ACCURACY: 0.9233 - 0.01s
Epoch [5/20] - loss: 0.3192 - ACCURACY: 0.9400 - 0.02s
Epoch [6/20] - loss: 0.2610 - ACCURACY: 0.9467 - 0.01s
Epoch [7/20] - loss: 0.2188 - ACCURACY: 0.9667 - 0.01s
Epoch [8/20] - loss: 0.1902 - ACCURACY: 0.9600 - 0.01s
Epoch [9/20] - loss: 0.1641 - ACCURACY: 0.9833 - 0.01s
Epoch [10/20] - loss: 0.1473 - ACCURACY: 0.9767 - 0.01s
Epoch [11/20] - loss: 0.1292 - ACCURACY: 0.9900 - 0.01s
Epoch [12/20] - loss: 0.1229 - ACCURACY: 0.9900 - 0.01s
Epoch [13/20] - loss: 0.1093 - ACCURACY: 0.9833 - 0.01s
Epoch [14/20] - loss: 0.1040 - ACCURACY: 0.9900 - 0.01s
Epoch [15/20] - loss: 0.0963 - ACCURACY: 0.9833 - 0.01s
Epoch [16/20] - loss: 0.0901 - ACCURACY: 0.9900 - 0.01s
Epoch [17/20] - loss: 0.0876 - ACCURACY: 0.9900 - 0.01s
Epoch [18/20] - loss: 0.0823 - ACCUR